## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier, early_stopping
import xgboost as xgb
from sklearn.metrics import average_precision_score

print("Libraries loaded.")

Libraries loaded.


## 2. Load Data

In [3]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity    = pd.read_csv("../data/train_identity.csv")

print("train_transaction:", train_transaction.shape)
print("train_identity:   ", train_identity.shape)

train_transaction: (590540, 394)
train_identity:    (144233, 41)


## 3. Merge

In [4]:
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
print("Merged:", train.shape)

Merged: (590540, 434)


## 4. Preprocessing
Done on the full `train` df before slicing so X_train/X_val inherit consistent dtypes and category levels.

In [5]:
# M1-M3,M5,M7-M9: T/F → 0/1
for col in ['M1','M2','M3','M5','M7','M8','M9']:
    train[col] = train[col].map({'T': 1, 'F': 0})

# Categorical columns — encode on full df so both splits share the same levels
for col in ['ProductCD','card4','card6','DeviceType',
            'id_30','id_31','P_emaildomain','R_emaildomain','M4','M6']:
    train[col] = train[col].astype('category')

# DeviceBrand: first token of DeviceInfo
train['DeviceBrand'] = (
    train['DeviceInfo']
    .fillna('missing')
    .str.split()
    .str[0]
    .astype('category')
)

print("Preprocessing done.")
print("DeviceBrand unique:", train['DeviceBrand'].nunique())

Preprocessing done.
DeviceBrand unique: 1183


## 5. Time-Based Train / Val Split

In [6]:
threshold = train['TransactionDT'].quantile(0.80)

X_train = train[train['TransactionDT'] < threshold].copy()
X_val   = train[train['TransactionDT'] >= threshold].copy()

print(f"X_train: {X_train.shape}  fraud rate: {X_train['isFraud'].mean():.4f}")
print(f"X_val:   {X_val.shape}  fraud rate: {X_val['isFraud'].mean():.4f}")

X_train: (472432, 435)  fraud rate: 0.0351
X_val:   (118108, 435)  fraud rate: 0.0344


## 6. Feature Engineering
All uid/aggregation features are computed from X_train only (no leakage). Val gets mapped with fillna fallback.

In [7]:
# ── uid1: card1 + addr1 ────────────────────────────────────────────────
X_train['uid'] = X_train['card1'].astype(str) + '_' + X_train['addr1'].astype(str)
X_val['uid']   = X_val['card1'].astype(str)   + '_' + X_val['addr1'].astype(str)

uid_count_map    = X_train.groupby('uid').size()
uid_amt_mean_map = X_train.groupby('uid')['TransactionAmt'].mean()
uid_amt_std_map  = X_train.groupby('uid')['TransactionAmt'].std()

X_train['uid_count']    = X_train['uid'].map(uid_count_map)
X_val['uid_count']      = X_val['uid'].map(uid_count_map).fillna(0)

X_train['uid_amt_mean'] = X_train['uid'].map(uid_amt_mean_map)
X_val['uid_amt_mean']   = X_val['uid'].map(uid_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

X_train['uid_amt_std']  = X_train['uid'].map(uid_amt_std_map).fillna(0)
X_val['uid_amt_std']    = X_val['uid'].map(uid_amt_std_map).fillna(0)

for col in ['C1','C2','C13','C14']:
    m = X_train.groupby('uid')[col].mean()
    X_train[f'uid_{col}_mean'] = X_train['uid'].map(m)
    X_val[f'uid_{col}_mean']   = X_val['uid'].map(m).fillna(X_train[col].mean())

# ── D1n ─────────────────────────────────────────────────────────────────
for df in [X_train, X_val]:
    df['D1n'] = np.floor(df['TransactionDT'] / (24*60*60)) - df['D1']

# ── uid2: card1+card2+card3+card5 ───────────────────────────────────────
for df in [X_train, X_val]:
    df['uid2'] = (df['card1'].astype(str) + '_' + df['card2'].astype(str) + '_' +
                  df['card3'].astype(str) + '_' + df['card5'].astype(str))

uid2_amt_mean_map = X_train.groupby('uid2')['TransactionAmt'].mean()
uid2_amt_std_map  = X_train.groupby('uid2')['TransactionAmt'].std()

X_train['uid2_amt_mean'] = X_train['uid2'].map(uid2_amt_mean_map)
X_val['uid2_amt_mean']   = X_val['uid2'].map(uid2_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

X_train['uid2_amt_std']  = X_train['uid2'].map(uid2_amt_std_map).fillna(0)
X_val['uid2_amt_std']    = X_val['uid2'].map(uid2_amt_std_map).fillna(0)

# ── uid_d1: card1 + normalised D1 day ───────────────────────────────────
for df in [X_train, X_val]:
    df['uid_d1'] = (df['card1'].astype(str) + '_' +
                    np.floor(df['TransactionDT'] / (24*60*60) - df['D1']).astype(str))

uid_d1_amt_mean_map = X_train.groupby('uid_d1')['TransactionAmt'].mean()
X_train['uid_d1_amt_mean'] = X_train['uid_d1'].map(uid_d1_amt_mean_map)
X_val['uid_d1_amt_mean']   = X_val['uid_d1'].map(uid_d1_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

print("Feature engineering done.")

Feature engineering done.


## 7. Final Feature List
Defined exactly once. Never modified after this cell.

In [8]:
features = [
    'TransactionAmt',
    'ProductCD',
    'card4',
    'card6',
    'DeviceType',
    'id_30',
    'id_31',
    'P_emaildomain',
    'R_emaildomain',
    'addr1',
    'addr2',
    'M4',
    'M6',
    'V188',
    'V189',
    'V200',
    'V201',
    'V242',
    'V244',
    'V246',
    'V257',
    'V258',
    'V147',
    'V148',
    'V149',
    'V154',
    'V155',
    'V156',
    'V157',
    'V158',
    'D1',
    'D2',
    'D3',
    'D4',
    'D5',
    'D10',
    'D11',
    'D15',
    'C1',
    'C2',
    'C3',
    'C4',
    'C5',
    'C6',
    'C7',
    'C8',
    'C9',
    'C10',
    'C11',
    'C12',
    'C13',
    'C14',
    'M1',
    'M2',
    'M3',
    'M5',
    'M7',
    'M8',
    'M9',
    'V39',
    'V40',
    'V42',
    'V43',
    'V44',
    'V45',
    'V170',
    'V171',
    'V46',
    'V47',
    'V48',
    'V49',
    'V50',
    'V51',
    'V52',
    'id_01',
    'id_02',
    'id_03',
    'id_04',
    'id_05',
    'id_06',
    'id_09',
    'id_10',
    'id_11',
    'id_13',
    'id_17',
    'id_20',
    'DeviceBrand',
    'uid_count',
    'uid_C1_mean',
    'uid_C2_mean',
    'uid_C13_mean',
    'uid_C14_mean',
    'uid_amt_mean',
    'uid_amt_std',
    'D1n',
    'uid2_amt_mean',
    'uid2_amt_std',
    'uid_d1_amt_mean',
]

# Sanity checks
assert len(features) == len(set(features)), "Duplicate features!"
missing = [f for f in features if f not in X_train.columns]
assert not missing, f"Missing from X_train: {missing}"
print(f"Features: {len(features)} — unique ✓ — all present in X_train ✓")

Features: 98 — unique ✓ — all present in X_train ✓


## 8. Categorical Alignment
Force X_val categories to match X_train exactly. Prevents LightGBM `categorical_feature do not match` error.

In [9]:
for col in features:
    if str(X_train[col].dtype) == 'category':
        X_val[col] = pd.Categorical(X_val[col], categories=X_train[col].cat.categories)

cat_in_features = [c for c in features if str(X_train[c].dtype) == 'category']
print(f"Categorical features ({len(cat_in_features)}):", cat_in_features)

Categorical features (11): ['ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'M4', 'M6', 'DeviceBrand']


## 9. Train LightGBM

In [10]:
model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=256,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_val[features], X_val['isFraud'])],
    eval_metric='average_precision',
    callbacks=[early_stopping(100)]
)

lgb_preds = model.predict_proba(X_val[features])[:, 1]
lgb_prauc = average_precision_score(X_val['isFraud'], lgb_preds)

print(f"LGB best iteration : {model.best_iteration_}")
print(f"LightGBM PR-AUC    : {lgb_prauc:.6f}")
assert len(model.feature_name_) == len(features), \
    f"Feature mismatch: model={len(model.feature_name_)} list={len(features)}"
print(f"Feature count check: {len(features)} ✓")

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10586
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1065]	valid_0's average_precision: 0.62005	valid_0's binary_logloss: 0.0786771
LGB best iteration : 1065
LightGBM PR-AUC    : 0.620050
Feature count check: 98 ✓


## 10. Train XGBoost

In [11]:
xgb_model = xgb.XGBClassifier(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=12,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.4,
    eval_metric='aucpr',
    random_state=42,
    tree_method='hist',
    enable_categorical=True
)

xgb_model.fit(
    X_train[features],
    X_train["isFraud"]
)

xgb_preds = xgb_model.predict_proba(X_val[features])[:, 1]
xgb_prauc = average_precision_score(X_val['isFraud'], xgb_preds)

print(
    "Best iteration:",
    getattr(xgb_model, "best_iteration", "Not using early stopping")
)
print(f"XGBoost PR-AUC     : {xgb_prauc:.6f}")
assert len(xgb_model.get_booster().feature_names) == len(features), \
    f"Feature mismatch: model={len(xgb_model.get_booster().feature_names)} list={len(features)}"
print(f"Feature count check: {len(features)} ✓")

Best iteration: Not using early stopping
XGBoost PR-AUC     : 0.617443
Feature count check: 98 ✓


## 11. Ensemble & Weight Search

In [12]:
# Both models already scored above — just combine
best_score = -1
best_w = 0.5

for w in np.arange(0.30, 0.71, 0.01):
    preds = w * xgb_preds + (1 - w) * lgb_preds
    score = average_precision_score(X_val['isFraud'], preds)
    if score > best_score:
        best_score = score
        best_w = w

ensemble_preds = best_w * xgb_preds + (1 - best_w) * lgb_preds
ensemble_prauc = average_precision_score(X_val['isFraud'], ensemble_preds)

print(f"LightGBM  PR-AUC : {lgb_prauc:.6f}")
print(f"XGBoost   PR-AUC : {xgb_prauc:.6f}")
print(f"Ensemble  PR-AUC : {ensemble_prauc:.6f}")
print(f"Best weight      : {best_w:.2f} XGB / {1-best_w:.2f} LGB")

LightGBM  PR-AUC : 0.620050
XGBoost   PR-AUC : 0.617443
Ensemble  PR-AUC : 0.631029
Best weight      : 0.53 XGB / 0.47 LGB


## 12. Consistency Check

In [13]:
n_feat    = len(features)
n_lgb     = len(model.feature_name_)
n_xgb     = len(xgb_model.get_booster().feature_names)

print(f"features list         : {n_feat}")
print(f"LightGBM feature_name_: {n_lgb}")
print(f"XGBoost feature_names : {n_xgb}")

assert n_feat == n_lgb == n_xgb, "MISMATCH — check feature engineering!"
print("\nAll counts match ✓")

features list         : 98
LightGBM feature_name_: 98
XGBoost feature_names : 98

All counts match ✓


## 13. Save Models

In [14]:
joblib.dump(model,     'lgb_final.pkl')
joblib.dump(xgb_model, 'xgb_final.pkl')
joblib.dump(features,  'features_final.pkl')

for fname in ['lgb_final.pkl','xgb_final.pkl','features_final.pkl']:
    size_mb = os.path.getsize(fname) / 1e6
    print(f"  {fname}: {size_mb:.1f} MB")

  lgb_final.pkl: 30.8 MB
  xgb_final.pkl: 53.7 MB
  features_final.pkl: 0.0 MB


In [15]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Use only numeric features (IF cannot handle category dtype)
numeric_features = [f for f in features
                    if str(X_train[f].dtype) not in ('category','object')]

X_train_if = X_train[numeric_features].fillna(X_train[numeric_features].median())
X_val_if   = X_val[numeric_features].fillna(X_train[numeric_features].median())

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.035,   # ~fraud rate in dataset
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_if)

# Convert to [0,1] where 1 = most anomalous
raw_scores = iso_model.decision_function(X_val_if)
iso_scores = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

from sklearn.metrics import average_precision_score
print('Isolation Forest PR-AUC:', average_precision_score(X_val['isFraud'], iso_scores))
print('(Low is expected - IF is unsupervised, it adds diversity not raw score)')

Isolation Forest PR-AUC: 0.2399203689434906
(Low is expected - IF is unsupervised, it adds diversity not raw score)


In [16]:
import joblib, os, json

joblib.dump(model,           'lgb_finals.pkl')
joblib.dump(xgb_model,       'xgb_finals.pkl')
joblib.dump(iso_model,       'iso_finals.pkl')
joblib.dump(features,        'features_finals.pkl')
joblib.dump(numeric_features,'numeric_features_finals.pkl')

# Save training medians for API imputation
medians = X_train[numeric_features].median().to_dict()
with open('train_medians.json','w') as f:
    json.dump(medians, f)

for fname in ['lgb_finals.pkl','xgb_finals.pkl','iso_finals.pkl',
              'features_finals.pkl','numeric_features_finals.pkl','train_medians.json']:
    size = os.path.getsize(fname)
    print(f'  {fname}: {size/1e6:.1f} MB')

  lgb_finals.pkl: 30.8 MB
  xgb_finals.pkl: 53.7 MB
  iso_finals.pkl: 1.4 MB
  features_finals.pkl: 0.0 MB
  numeric_features_finals.pkl: 0.0 MB
  train_medians.json: 0.0 MB


In [17]:
from collections import deque

class AdaptiveThresholdEngine:
    """
    Tracks rolling fraud rate over last N transactions.
    Tightens threshold when fraud rate spikes above expected.
    """
    def __init__(self,
                 base_threshold: float = 0.50,
                 expected_fraud_rate: float = 0.035,
                 window: int = 1000,
                 min_threshold: float = 0.25,
                 max_threshold: float = 0.70):
        self.base_threshold  = base_threshold
        self.expected_rate   = expected_fraud_rate
        self.window          = window
        self.min_threshold   = min_threshold
        self.max_threshold   = max_threshold
        self._recent_labels  = deque(maxlen=window)

    def update(self, is_fraud: int):
        self._recent_labels.append(is_fraud)

    @property
    def current_threshold(self) -> float:
        if len(self._recent_labels) < 50:
            return self.base_threshold
        observed_rate = sum(self._recent_labels) / len(self._recent_labels)
        ratio = observed_rate / max(self.expected_rate, 1e-6)
        if ratio > 2.0:   t = self.base_threshold * 0.70
        elif ratio > 1.5: t = self.base_threshold * 0.85
        elif ratio < 0.5: t = self.base_threshold * 1.15
        else:             t = self.base_threshold
        return float(np.clip(t, self.min_threshold, self.max_threshold))

    def should_flag(self, score: float) -> bool:
        return score >= self.current_threshold

    def __repr__(self):
        n    = len(self._recent_labels)
        rate = sum(self._recent_labels) / max(n, 1)
        return (f'AdaptiveThresholdEngine('
                f'threshold={self.current_threshold:.3f}, '
                f'observed_rate={rate:.3f}, window={n})')

# Tests
eng = AdaptiveThresholdEngine()
print('Fresh:          ', eng.current_threshold)
for _ in range(800): eng.update(0)
for _ in range(20):  eng.update(1)
print('Low fraud:      ', eng.current_threshold)
for _ in range(100): eng.update(1)
print('After spike:    ', eng.current_threshold)
print(eng)

Fresh:           0.5
Low fraud:       0.5
After spike:     0.35
AdaptiveThresholdEngine(threshold=0.350, observed_rate=0.130, window=920)


In [18]:
class BusinessRulesLayer: 
    """
    Hard rules that fire regardless of ML score.
    Returns (flagged: bool, reasons: list[str])
    """
    HIGH_RISK_EMAILS = {
        'protonmail.com','mail.com','guerrillamail.com',
        'tempmail.com','throwam.com'
    }

    def evaluate(self, score, amount, hour,
                 email_domain='', adaptive_threshold=0.50):
        reasons = []
        flagged = False

        if amount > 50_000 and score > 0.30:
            reasons.append(f'HIGH_AMOUNT: {amount:,.0f} + score {score:.3f}')
            flagged = True

        if 2 <= hour <= 5 and score > 0.25:
            reasons.append(f'NIGHT_TXN: hour={hour}, score={score:.3f}')
            flagged = True

        if email_domain.lower() in self.HIGH_RISK_EMAILS and score > 0.20:
            reasons.append(f'RISKY_EMAIL: {email_domain}, score={score:.3f}')
            flagged = True

        if score >= adaptive_threshold:
            reasons.append(f'ML_SCORE: {score:.3f} >= threshold {adaptive_threshold:.3f}')
            flagged = True

        return flagged, reasons

rules = BusinessRulesLayer()
print('Large amount:', rules.evaluate(0.32, 75000, 14, 'gmail.com'))
print('Night txn:   ', rules.evaluate(0.28, 5000,  3,  'gmail.com'))
print('Risky email: ', rules.evaluate(0.22, 5000,  14, 'protonmail.com'))
print('Clean:       ', rules.evaluate(0.15, 500,   14, 'gmail.com'))

Large amount: (True, ['HIGH_AMOUNT: 75,000 + score 0.320'])
Night txn:    (True, ['NIGHT_TXN: hour=3, score=0.280'])
Risky email:  (True, ['RISKY_EMAIL: protonmail.com, score=0.220'])
Clean:        (False, [])


In [19]:
from dataclasses import dataclass, field
import time

@dataclass
class ScoredTransaction:
    transaction_id: str
    lgb_score:      float
    xgb_score:      float
    iso_score:      float
    ensemble_score: float
    threshold:      float
    is_flagged:     bool
    reasons:        list
    latency_ms:     float = 0.0
    timestamp:      float = field(default_factory=time.time)

    def summary(self) -> str:
        flag = '🚨 FRAUD' if self.is_flagged else '✅ LEGIT'
        return (f'{flag} | txn={self.transaction_id} | '
                f'score={self.ensemble_score:.3f} | '
                f'threshold={self.threshold:.3f} | '
                f'latency={self.latency_ms:.1f}ms | '
                f'reasons={self.reasons}')

# Test
txn = ScoredTransaction(
    transaction_id='TXN_001', lgb_score=0.72, xgb_score=0.68,
    iso_score=0.55, ensemble_score=0.69, threshold=0.50,
    is_flagged=True,
    reasons=['ML_SCORE: 0.69 >= threshold 0.50', 'NIGHT_TXN: hour=3'],
    latency_ms=4.2
)
print(txn.summary())

🚨 FRAUD | txn=TXN_001 | score=0.690 | threshold=0.500 | latency=4.2ms | reasons=['ML_SCORE: 0.69 >= threshold 0.50', 'NIGHT_TXN: hour=3']


In [20]:
import time

_lgb         = joblib.load('lgb_finals.pkl')
_xgb         = joblib.load('xgb_finals.pkl')
_feats       = joblib.load('features_finals.pkl')

_engine      = AdaptiveThresholdEngine()
_rules       = BusinessRulesLayer()

def score_transaction(row: dict) -> ScoredTransaction:
    t0  = time.perf_counter()
    df  = pd.DataFrame([row])

    # Align category dtypes
    for col in _feats:
        if col in df.columns and col in X_train.columns:
            if str(X_train[col].dtype) == 'category':
                df[col] = pd.Categorical(df[col],
                              categories=X_train[col].cat.categories)

    lgb_s = float(_lgb.predict_proba(df[_feats])[:, 1][0])
    xgb_s = float(_xgb.predict_proba(df[_feats])[:, 1][0])

    

    ens_s     = 0.47 * lgb_s + 0.53 * xgb_s
    threshold = _engine.current_threshold
    amount    = row.get('TransactionAmt', 0)
    hour      = int((row.get('TransactionDT', 0) // 3600) % 24)
    email     = str(row.get('P_emaildomain', ''))

    flagged, reasons = _rules.evaluate(
        score=ens_s, amount=amount, hour=hour,
        email_domain=email, adaptive_threshold=threshold
    )
    latency = (time.perf_counter() - t0) * 1000

    return ScoredTransaction(
        transaction_id=str(row.get('TransactionID','?')),
        lgb_score=lgb_s, xgb_score=xgb_s, iso_score=0.0,  # Placeholder since IF is not used in this API version
        ensemble_score=ens_s, threshold=threshold,
        is_flagged=flagged, reasons=reasons, latency_ms=latency
    )

# Test
result = score_transaction(X_val.iloc[0].to_dict())
print(result.summary())

✅ LEGIT | txn=3459432 | score=0.071 | threshold=0.500 | latency=20.5ms | reasons=[]


In [21]:
for _ in range(5):
    print(score_transaction(X_val.iloc[0].to_dict()).latency_ms)

18.24908300000061
15.650625000006357
15.591542000009895
14.957457999997814
13.478167000016583


In [22]:
for _ in range(5):
    print(score_transaction(X_val.iloc[0].to_dict()).latency_ms)

17.943666000007852
13.913041999984443
13.972958000010749
13.47770800001058
13.673417000006793


In [23]:
for _ in range(5):
    print(score_transaction(X_val.iloc[0].to_dict()).latency_ms)

17.350582999995368
14.46558399999276
13.898167000007788
15.169667000009213
14.668000000000347


In [24]:
import time, numpy as np

N = 1000
rows = X_val.head(N).to_dict(orient='records')

t0      = time.perf_counter()
results = [score_transaction(r) for r in rows]
elapsed = time.perf_counter() - t0

lats    = [r.latency_ms for r in results]
flagged = sum(r.is_flagged for r in results)

print(f'Scored {N} transactions in {elapsed:.2f}s')
print(f'Throughput  : {N/elapsed:,.0f} TPS')
print(f'Mean latency: {np.mean(lats):.2f} ms')
print(f'P50 latency : {np.percentile(lats,50):.2f} ms')
print(f'P95 latency : {np.percentile(lats,95):.2f} ms')
print(f'P99 latency : {np.percentile(lats,99):.2f} ms')
print(f'Flagged     : {flagged}/{N} ({100*flagged/N:.1f}%)')

Scored 1000 transactions in 13.86s
Throughput  : 72 TPS
Mean latency: 13.85 ms
P50 latency : 13.49 ms
P95 latency : 15.02 ms
P99 latency : 18.69 ms
Flagged     : 36/1000 (3.6%)


In [25]:
import time

row = X_val.iloc[5].to_dict()
df  = pd.DataFrame([row])

t0 = time.perf_counter()
for col in _feats:
    if col in df.columns and col in X_train.columns:
        if str(X_train[col].dtype) == 'category':
            df[col] = pd.Categorical(df[col], categories=X_train[col].cat.categories)
print(f"Category alignment : {(time.perf_counter()-t0)*1000:.2f} ms")

t0 = time.perf_counter()
_lgb.predict_proba(df[_feats])
print(f"LGB predict        : {(time.perf_counter()-t0)*1000:.2f} ms")

t0 = time.perf_counter()
_xgb.predict_proba(df[_feats])
print(f"XGB predict        : {(time.perf_counter()-t0)*1000:.2f} ms")

Category alignment : 1.67 ms
LGB predict        : 4.16 ms
XGB predict        : 9.70 ms


In [26]:
_cat_cols = {
    col: X_train[col].cat.categories
    for col in _feats
    if col in X_train.columns and str(X_train[col].dtype) == 'category'
}

print("done, cat cols:", list(_cat_cols.keys()))

done, cat cols: ['ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'M4', 'M6', 'DeviceBrand']


In [27]:
import time, numpy as np

rows = X_val.head(1000).to_dict(orient='records')

# Build one big table of all 1000 rows at once
df = pd.DataFrame(rows)

# Fix category columns
for col, cats in _cat_cols.items():
    if col in df.columns:
        df[col] = pd.Categorical(df[col], categories=cats)

# Score ALL 1000 in one shot — not one by one
t0 = time.perf_counter()
lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
ens_all = 0.47 * lgb_all + 0.53 * xgb_all
elapsed = time.perf_counter() - t0

flagged = sum(ens_all > 0.5)

print(f'Scored 1000 transactions in {elapsed:.2f}s')
print(f'Throughput  : {1000/elapsed:,.0f} TPS')
print(f'Mean latency: {elapsed/1000*1000:.2f} ms')
print(f'Flagged     : {flagged}/1000 ({100*flagged/1000:.1f}%)')

Scored 1000 transactions in 0.07s
Throughput  : 13,367 TPS
Mean latency: 0.07 ms
Flagged     : 33/1000 (3.3%)


In [28]:
for n in [1000, 5000, 10000, 50000]:
    rows = X_val.head(n).to_dict(orient='records')
    df = pd.DataFrame(rows)
    for col, cats in _cat_cols.items():
        if col in df.columns:
            df[col] = pd.Categorical(df[col], categories=cats)
    t0 = time.perf_counter()
    lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
    xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
    elapsed = time.perf_counter() - t0
    print(f'{n:>6} rows → {n/elapsed:,.0f} TPS')

  1000 rows → 12,663 TPS
  5000 rows → 16,058 TPS
 10000 rows → 17,463 TPS
 50000 rows → 17,173 TPS


In [29]:
import asyncio, time, numpy as np
from collections import deque

class StreamMonitor:
    def __init__(self, window=500):
        self._lats     = deque(maxlen=window)
        self._flags    = deque(maxlen=window)
        self.total     = 0
        self.n_flagged = 0
        self._t0       = time.perf_counter()

    def record(self, r: ScoredTransaction):
        self._lats.append(r.latency_ms)
        self._flags.append(int(r.is_flagged))
        self.total     += 1
        self.n_flagged += int(r.is_flagged)

    @property
    def tps(self):
        return self.total / max(time.perf_counter() - self._t0, 1e-6)

    @property
    def live_fraud_rate(self):
        return sum(self._flags) / max(len(self._flags), 1)

    @property
    def p95_latency(self):
        return float(np.percentile(list(self._lats), 95)) if self._lats else 0

    def summary(self):
        return (f'TPS={self.tps:,.0f} | '
                f'fraud_rate={self.live_fraud_rate:.3f} | '
                f'p95={self.p95_latency:.1f}ms | '
                f'total={self.total}')


async def producer(q, rows, speed=500):
    delay = 1.0 / speed
    for row in rows:
        await q.put(row)
        await asyncio.sleep(delay)
    await q.put(None)

async def consumer(q, monitor):
    flagged_log = []
    batch = []

    while True:
        row = await q.get()

        if row is None:
            # score whatever is left in the batch
            if batch:
                df = pd.DataFrame(batch)
                for col, cats in _cat_cols.items():
                    if col in df.columns:
                        df[col] = pd.Categorical(df[col], categories=cats)
                lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
                xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
                ens_all = 0.47 * lgb_all + 0.53 * xgb_all
                for i, r in enumerate(batch):
                    ens_s = float(ens_all[i])
                    threshold = _engine.current_threshold
                    flagged, reasons = _rules.evaluate(
                        ens_s, r.get('TransactionAmt',0),
                        int((r.get('TransactionDT',0)//3600)%24),
                        str(r.get('P_emaildomain','')), threshold
                    )
                    result = ScoredTransaction(
                        transaction_id=str(r.get('TransactionID','?')),
                        lgb_score=float(lgb_all[i]),
                        xgb_score=float(xgb_all[i]),
                        iso_score=0.0, ensemble_score=ens_s,
                        threshold=threshold, is_flagged=flagged,
                        reasons=reasons, latency_ms=0.0
                    )
                    monitor.record(result)
                    if result.is_flagged:
                        flagged_log.append(result)
            break

        batch.append(row)

        # score every 50 rows
        if len(batch) >= 50:
            df = pd.DataFrame(batch)
            for col, cats in _cat_cols.items():
                if col in df.columns:
                    df[col] = pd.Categorical(df[col], categories=cats)
            lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
            xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
            ens_all = 0.47 * lgb_all + 0.53 * xgb_all
            for i, r in enumerate(batch):
                ens_s = float(ens_all[i])
                threshold = _engine.current_threshold
                flagged, reasons = _rules.evaluate(
                    ens_s, r.get('TransactionAmt',0),
                    int((r.get('TransactionDT',0)//3600)%24),
                    str(r.get('P_emaildomain','')), threshold
                )
                result = ScoredTransaction(
                    transaction_id=str(r.get('TransactionID','?')),
                    lgb_score=float(lgb_all[i]),
                    xgb_score=float(xgb_all[i]),
                    iso_score=0.0, ensemble_score=ens_s,
                    threshold=threshold, is_flagged=flagged,
                    reasons=reasons, latency_ms=0.0
                )
                monitor.record(result)
                if result.is_flagged:
                    flagged_log.append(result)
            batch = []
        q.task_done()

    return flagged_log

async def run_stream(rows, speed=500):
    q = asyncio.Queue(maxsize=200)
    m = StreamMonitor()
    prod = asyncio.create_task(producer(q, rows, speed))
    cons = asyncio.create_task(consumer(q, m))
    await prod
    flagged = await cons
    return m, flagged

# Run
stream_rows = X_val.head(2000).to_dict(orient='records')
monitor, flagged_txns = await run_stream(stream_rows, speed=500)

print('=' * 50)
print('STREAMING BENCHMARK')
print('=' * 50)
print(f'Transactions : {monitor.total:,}')
print(f'Throughput   : {monitor.tps:,.0f} TPS')
print(f'Flagged      : {monitor.n_flagged} ({100*monitor.n_flagged/monitor.total:.1f}%)')
print(f'P95 latency  : {monitor.p95_latency:.1f} ms')
print()
for t in flagged_txns[:3]:
    print(' ', t.summary())

STREAMING BENCHMARK
Transactions : 2,000
Throughput   : 318 TPS
Flagged      : 55 (2.8%)
P95 latency  : 0.0 ms

  🚨 FRAUD | txn=3459499 | score=0.374 | threshold=0.500 | latency=0.0ms | reasons=['NIGHT_TXN: hour=3, score=0.374']
  🚨 FRAUD | txn=3459504 | score=0.431 | threshold=0.500 | latency=0.0ms | reasons=['NIGHT_TXN: hour=4, score=0.431']
  🚨 FRAUD | txn=3459510 | score=0.885 | threshold=0.500 | latency=0.0ms | reasons=['NIGHT_TXN: hour=4, score=0.885', 'ML_SCORE: 0.885 >= threshold 0.500']


In [30]:
app_code = 'import asyncio, sqlite3, time, json\nimport joblib, numpy as np, pandas as pd\nimport uvicorn\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom pydantic import BaseModel\n\n# ── Load models ────────────────────────────────────────────────────────────────\n_lgb        = joblib.load("lgb_final.pkl")\n_xgb        = joblib.load("xgb_final.pkl")\n_iso        = joblib.load("iso_final.pkl")\n_feats      = joblib.load("features_final.pkl")\n_num_feats  = joblib.load("numeric_features_final.pkl")\nwith open("train_medians.json") as f:\n    _meds = pd.Series(json.load(f))\n\nW_LGB, W_XGB, W_ISO = 0.50, 0.35, 0.15\nDB = "flagged.db"\n\ndef init_db():\n    con = sqlite3.connect(DB)\n    con.execute("CREATE TABLE IF NOT EXISTS flagged (id INTEGER PRIMARY KEY AUTOINCREMENT, txn_id TEXT, score REAL, threshold REAL, reasons TEXT, latency_ms REAL, reviewed INTEGER DEFAULT 0, label TEXT, ts REAL)")\n    con.commit(); con.close()\ninit_db()\n\nfrom collections import deque\n\nclass _ThresholdEngine:\n    def __init__(self):\n        self._buf = deque(maxlen=1000)\n    def update(self, v): self._buf.append(v)\n    @property\n    def threshold(self):\n        if len(self._buf) < 50: return 0.50\n        r = sum(self._buf)/len(self._buf)/0.035\n        if r > 2.0:   return 0.35\n        elif r > 1.5: return 0.42\n        elif r < 0.5: return 0.58\n        return 0.50\n\nclass _Monitor:\n    def __init__(self):\n        self.total=0; self.flagged=0\n        self._lats=deque(maxlen=500); self._t0=time.perf_counter()\n    def record(self, lat, flag):\n        self.total+=1; self.flagged+=flag; self._lats.append(lat)\n    @property\n    def tps(self): return self.total/max(time.perf_counter()-self._t0,1e-6)\n    @property\n    def p95(self): return float(np.percentile(list(self._lats),95)) if self._lats else 0\n\n_eng = _ThresholdEngine()\n_mon = _Monitor()\n_ws_clients = []\n\nHIGH_RISK = {\'protonmail.com\',\'mail.com\',\'guerrillamail.com\'}\n\napp = FastAPI(title="Fraud Detection Engine", version="1.0")\n\nclass TxnIn(BaseModel):\n    TransactionID: str = "?"\n    TransactionAmt: float = 0\n    TransactionDT: int = 0\n    P_emaildomain: str = ""\n    model_config = {"extra":"allow"}\n\nclass ReviewIn(BaseModel):\n    label: str\n\ndef _score(row):\n    t0  = time.perf_counter()\n    df  = pd.DataFrame([row])\n    lgb_s = float(_lgb.predict_proba(df[_feats])[:,1][0])\n    xgb_s = float(_xgb.predict_proba(df[_feats])[:,1][0])\n    raw   = _iso.decision_function(df[_num_feats].fillna(_meds))[0]\n    iso_s = float(np.clip(1-(raw+0.5),0,1))\n    ens   = W_LGB*lgb_s + W_XGB*xgb_s + W_ISO*iso_s\n    thr   = _eng.threshold\n    amt   = row.get("TransactionAmt",0)\n    hr    = int((row.get("TransactionDT",0)//3600)%24)\n    em    = str(row.get("P_emaildomain","")).lower()\n    reasons=[]\n    flag=False\n    if amt>50000 and ens>0.30: reasons.append(f"HIGH_AMOUNT:{amt:.0f}"); flag=True\n    if 2<=hr<=5 and ens>0.25:  reasons.append(f"NIGHT_TXN:hour={hr}"); flag=True\n    if em in HIGH_RISK and ens>0.20: reasons.append(f"RISKY_EMAIL:{em}"); flag=True\n    if ens>=thr: reasons.append(f"ML_SCORE:{ens:.3f}"); flag=True\n    lat=(time.perf_counter()-t0)*1000\n    return {"txn_id":row.get("TransactionID","?"),"lgb":lgb_s,"xgb":xgb_s,\n            "iso":iso_s,"score":ens,"threshold":thr,"flagged":flag,\n            "reasons":reasons,"latency_ms":lat}\n\n@app.post("/score")\nasync def score(txn: TxnIn):\n    r = _score(txn.model_dump())\n    _mon.record(r["latency_ms"], r["flagged"])\n    if r["flagged"]:\n        con=sqlite3.connect(DB)\n        con.execute("INSERT INTO flagged(txn_id,score,threshold,reasons,latency_ms,ts) VALUES(?,?,?,?,?,?)",\n                    (r["txn_id"],r["score"],r["threshold"],str(r["reasons"]),r["latency_ms"],time.time()))\n        con.commit(); con.close()\n        for ws in _ws_clients:\n            try: await ws.send_json(r)\n            except: pass\n    return r\n\n@app.get("/transactions/flagged")\nasync def get_flagged(limit:int=50, offset:int=0):\n    con=sqlite3.connect(DB)\n    rows=con.execute("SELECT * FROM flagged ORDER BY ts DESC LIMIT ? OFFSET ?",(limit,offset)).fetchall()\n    con.close()\n    cols=["id","txn_id","score","threshold","reasons","latency_ms","reviewed","label","ts"]\n    return [dict(zip(cols,r)) for r in rows]\n\n@app.post("/transactions/{txn_id}/review")\nasync def review(txn_id:str, body:ReviewIn):\n    con=sqlite3.connect(DB)\n    con.execute("UPDATE flagged SET reviewed=1,label=? WHERE txn_id=?",(body.label,txn_id))\n    con.commit(); con.close()\n    _eng.update(1 if body.label=="fraud" else 0)\n    return {"status":"ok"}\n\n@app.get("/metrics/live")\nasync def metrics():\n    return {"tps":round(_mon.tps,1),"total":_mon.total,\n            "flagged":_mon.flagged,"p95_ms":round(_mon.p95,2),\n            "threshold":round(_eng.threshold,4)}\n\n@app.websocket("/ws/live-flags")\nasync def ws_live(ws:WebSocket):\n    await ws.accept(); _ws_clients.append(ws)\n    try:\n        while True: await ws.receive_text()\n    except WebSocketDisconnect: _ws_clients.remove(ws)\n\nif __name__=="__main__":\n    uvicorn.run("app:app",host="0.0.0.0",port=8000,reload=False)\n'

with open("app.py","w") as f:
    f.write(app_code)

print("app.py written.")
print("Run with: pip install fastapi uvicorn && python app.py")
print("Endpoints:")
print("  POST /score              — score a transaction")
print("  GET  /transactions/flagged — list flagged txns")
print("  POST /transactions/{id}/review — analyst review")
print("  GET  /metrics/live       — live TPS + latency")
print("  WS   /ws/live-flags      — real-time push")

app.py written.
Run with: pip install fastapi uvicorn && python app.py
Endpoints:
  POST /score              — score a transaction
  GET  /transactions/flagged — list flagged txns
  POST /transactions/{id}/review — analyst review
  GET  /metrics/live       — live TPS + latency
  WS   /ws/live-flags      — real-time push


In [31]:
sample = X_val.iloc[0].to_dict()

In [32]:
sample

{'TransactionID': 3459432,
 'isFraud': 1,
 'TransactionDT': 12192900,
 'TransactionAmt': 33.261,
 'ProductCD': 'C',
 'card1': 9300,
 'card2': 103.0,
 'card3': 185.0,
 'card4': 'visa',
 'card5': 138.0,
 'card6': 'debit',
 'addr1': nan,
 'addr2': nan,
 'dist1': nan,
 'dist2': nan,
 'P_emaildomain': 'gmail.com',
 'R_emaildomain': 'gmail.com',
 'C1': 1.0,
 'C2': 1.0,
 'C3': 0.0,
 'C4': 1.0,
 'C5': 0.0,
 'C6': 1.0,
 'C7': 1.0,
 'C8': 1.0,
 'C9': 0.0,
 'C10': 1.0,
 'C11': 1.0,
 'C12': 1.0,
 'C13': 1.0,
 'C14': 1.0,
 'D1': 0.0,
 'D2': nan,
 'D3': nan,
 'D4': 0.0,
 'D5': nan,
 'D6': 0.0,
 'D7': nan,
 'D8': nan,
 'D9': nan,
 'D10': 0.0,
 'D11': nan,
 'D12': 0.0,
 'D13': 0.0,
 'D14': 0.0,
 'D15': 0.0,
 'M1': nan,
 'M2': nan,
 'M3': nan,
 'M4': 'M2',
 'M5': nan,
 'M6': nan,
 'M7': nan,
 'M8': nan,
 'M9': nan,
 'V1': nan,
 'V2': nan,
 'V3': nan,
 'V4': nan,
 'V5': nan,
 'V6': nan,
 'V7': nan,
 'V8': nan,
 'V9': nan,
 'V10': nan,
 'V11': nan,
 'V12': 0.0,
 'V13': 0.0,
 'V14': 1.0,
 'V15': 1.0,
 'V1

In [33]:
len(sample)


449

In [34]:
len(features)

98

In [35]:
import json

sample = X_val.iloc[0].to_dict()
sample.pop("isFraud", None)   # optional

print(json.dumps(sample, default=str, indent=2))

{
  "TransactionID": 3459432,
  "TransactionDT": 12192900,
  "TransactionAmt": 33.261,
  "ProductCD": "C",
  "card1": 9300,
  "card2": 103.0,
  "card3": 185.0,
  "card4": "visa",
  "card5": 138.0,
  "card6": "debit",
  "addr1": NaN,
  "addr2": NaN,
  "dist1": NaN,
  "dist2": NaN,
  "P_emaildomain": "gmail.com",
  "R_emaildomain": "gmail.com",
  "C1": 1.0,
  "C2": 1.0,
  "C3": 0.0,
  "C4": 1.0,
  "C5": 0.0,
  "C6": 1.0,
  "C7": 1.0,
  "C8": 1.0,
  "C9": 0.0,
  "C10": 1.0,
  "C11": 1.0,
  "C12": 1.0,
  "C13": 1.0,
  "C14": 1.0,
  "D1": 0.0,
  "D2": NaN,
  "D3": NaN,
  "D4": 0.0,
  "D5": NaN,
  "D6": 0.0,
  "D7": NaN,
  "D8": NaN,
  "D9": NaN,
  "D10": 0.0,
  "D11": NaN,
  "D12": 0.0,
  "D13": 0.0,
  "D14": 0.0,
  "D15": 0.0,
  "M1": NaN,
  "M2": NaN,
  "M3": NaN,
  "M4": "M2",
  "M5": NaN,
  "M6": NaN,
  "M7": NaN,
  "M8": NaN,
  "M9": NaN,
  "V1": NaN,
  "V2": NaN,
  "V3": NaN,
  "V4": NaN,
  "V5": NaN,
  "V6": NaN,
  "V7": NaN,
  "V8": NaN,
  "V9": NaN,
  "V10": NaN,
  "V11": NaN,
  "V

In [36]:
print(json.dumps(sample, default=str, indent=2)[:500])


{
  "TransactionID": 3459432,
  "TransactionDT": 12192900,
  "TransactionAmt": 33.261,
  "ProductCD": "C",
  "card1": 9300,
  "card2": 103.0,
  "card3": 185.0,
  "card4": "visa",
  "card5": 138.0,
  "card6": "debit",
  "addr1": NaN,
  "addr2": NaN,
  "dist1": NaN,
  "dist2": NaN,
  "P_emaildomain": "gmail.com",
  "R_emaildomain": "gmail.com",
  "C1": 1.0,
  "C2": 1.0,
  "C3": 0.0,
  "C4": 1.0,
  "C5": 0.0,
  "C6": 1.0,
  "C7": 1.0,
  "C8": 1.0,
  "C9": 0.0,
  "C10": 1.0,
  "C11": 1.0,
  "C12": 1


In [37]:
cat_cols = {}

for col in X_train.columns:
    if str(X_train[col].dtype) == "category":
        cat_cols[col] = list(X_train[col].cat.categories)

In [38]:
import json

with open("cat_cols.json", "w") as f:
    json.dump(cat_cols, f)

In [39]:
import shap

# TreeExplainer on LGB (fastest for tree models)
explainer = shap.TreeExplainer(model)
sample_df  = X_val[features].head(200)
shap_vals  = explainer.shap_values(sample_df)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
print('SHAP values computed. Shape:', sv.shape)

def explain_row(row_df, top_n=3):
    vals = explainer.shap_values(row_df)
    sv   = vals[1][0] if isinstance(vals, list) else vals[0]
    pairs = sorted(zip(features, sv), key=lambda x: abs(x[1]), reverse=True)
    return [{'feature':f,'value':float(row_df[f].iloc[0]),
             'shap':round(float(s),4),
             'direction':'up fraud' if s>0 else 'down fraud'}
            for f,s in pairs[:top_n]]

# Demo on a high-score transaction
high_score_idx = final_preds.argmax()
row_df = X_val[features].iloc[[high_score_idx]].reset_index(drop=True)
expl = explain_row(row_df)
print(f'Transaction score: {final_preds[high_score_idx]:.4f}')
print('Top SHAP contributors:')
for e in expl:
    print(f"  {e['feature']:20s}  val={e['value']:.3f}  "
          f"shap={e['shap']:+.4f}  {e['direction']}")

SHAP values computed. Shape: (200, 98)


NameError: name 'final_preds' is not defined